In [ ]:
# Colab runtime / NVIDIA GPU inventory (display only)
# This cell reports Colab hardware/runtime details only. It does not change training config.
import os
import sys
import platform
import shutil
import subprocess
from pathlib import Path

IS_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ

if not IS_COLAB:
    print("Not running in Colab; skipping Colab GPU inventory.")
else:
    def _run_cmd(cmd, timeout=20):
        exe = cmd[0]
        if shutil.which(exe) is None:
            return f"{exe} not found"
        try:
            result = subprocess.run(
                cmd,
                text=True,
                capture_output=True,
                timeout=timeout,
                check=False,
            )
            output = (result.stdout or result.stderr or "").strip()
            return output if output else f"{exe} returned no output"
        except Exception as exc:
            return f"{exe} failed: {type(exc).__name__}: {exc}"

    print(f"python: {sys.version.split()[0]}")
    print(f"platform: {platform.platform()}")
    print(f"cwd: {Path.cwd()}")
    print(f"colab env: {IS_COLAB}")

    try:
        import torch

        print(f"torch: {torch.__version__}")
        print(f"cuda available: {torch.cuda.is_available()}")
        print(f"cuda version: {torch.version.cuda}")
        print(f"gpu count: {torch.cuda.device_count()}")
        for gpu_index in range(torch.cuda.device_count()):
            props = torch.cuda.get_device_properties(gpu_index)
            print(f"gpu {gpu_index} name: {props.name}")
            print(f"gpu {gpu_index} total memory GB: {props.total_memory / 1024**3:.2f}")
            print(f"gpu {gpu_index} capability: {props.major}.{props.minor}")
            print(f"gpu {gpu_index} multiprocessors: {props.multi_processor_count}")
    except Exception as exc:
        print(f"torch gpu inspection failed: {type(exc).__name__}: {exc}")

    print("\n--- nvidia-smi gpu summary ---")
    print(_run_cmd([
        "nvidia-smi",
        "--query-gpu=index,name,memory.total,memory.used,driver_version,pstate,temperature.gpu,power.draw,power.limit",
        "--format=csv,noheader",
    ]))

    print("\n--- nvidia-smi full ---")
    print(_run_cmd(["nvidia-smi"]))

    print("\n--- disks ---")
    print(_run_cmd(["df", "-h"]))

    print("\n--- memory ---")
    print(_run_cmd(["free", "-h"]))


# Colab Train And Submit

Simple Colab workflow: load data, run baseline, train LoRA, package `submission.zip`.

All settings are in the next cell.

## 1. Config

In [ ]:
from pathlib import Path

# Possible MODEL_NAME values:
# "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16" # challenge target
# "google/gemma-4-E2B-it"                      # smaller test
# "google/gemma-4-E4B-it"                      # heavier Gemma test
MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

PROJECT_ROOT = Path("/content/nemotron_challenge")
# Keep data local, but write training outputs/checkpoints to Drive when enabled.
USE_GOOGLE_DRIVE_BACKUP = True
DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/nemotron_challenge")
OUTPUT_PROJECT_ROOT = DRIVE_PROJECT_ROOT if USE_GOOGLE_DRIVE_BACKUP else PROJECT_ROOT
AUTO_RESUME_FROM_CHECKPOINT = True
RESUME_CHECKPOINT = None  # Example: Path("/content/drive/MyDrive/nemotron_challenge/outputs/.../checkpoint-96")
FORCE_FRESH_RUN = False  # Set True only when intentionally discarding old checkpoints for this EXPERIMENT_NAME.
RAW_DIR = PROJECT_ROOT / "data" / "raw"
ZIP_PATH = Path("/content/nvidia-nemotron-model-reasoning-challenge.zip")
EXPERIMENT_NAME = "S4_attention_expand_r8_private_boxed_max128_drive"
OUTPUT_DIR = OUTPUT_PROJECT_ROOT / "outputs" / EXPERIMENT_NAME
ADAPTER_DIR = OUTPUT_DIR / "adapter"
SANITY_CSV_PATH = OUTPUT_DIR / "sanity_test_predictions.csv"
SANITY_RAW_CSV_PATH = OUTPUT_DIR / "sanity_test_predictions_raw.csv"
SUBMISSION_ZIP_PATH = OUTPUT_PROJECT_ROOT / f"{EXPERIMENT_NAME}_submission.zip"
RUN_CONFIG_PATH = OUTPUT_DIR / "run_config.json"
PROBE_SET_PATH = OUTPUT_DIR / "probe_questions.csv"
PROBE_EVOLUTION_CSV_PATH = OUTPUT_DIR / "probe_evolution.csv"
PROBE_EVOLUTION_JSONL_PATH = OUTPUT_DIR / "probe_evolution.jsonl"
TRAINER_LOG_CSV_PATH = OUTPUT_DIR / "trainer_log_history.csv"
NOTEBOOK_SNAPSHOT_PATH = PROJECT_ROOT / "04_colab_train_and_submit.ipynb"
DIAGNOSTICS_ZIP_PATH = OUTPUT_PROJECT_ROOT / f"{EXPERIMENT_NAME}_diagnostics.zip"

# Start small for Nemotron training. Raise to 512/1024 after one step works.
MAX_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 128
# Use None for the full training split. Keep EVAL_ROWS > 0 for diagnostics.
TRAIN_ROWS = None
EVAL_ROWS = 256
PROBE_ROWS = 5
LOG_EVAL_POINTS = 8
SAVE_POINTS = 4

PROMPT_EXPERIMENT = "private_reasoning_boxed"
SYSTEM_PROMPTS = {
    "direct_raw_0_62": "Solve the puzzle and output only the final answer value. Do not explain. Do not write prefixes like 'answer:' or 'the answer is'.",
    "boxed_final": "Solve the puzzle carefully. Output only the final answer inside \\boxed{} with no trailing text.",
    "private_reasoning_boxed": "Solve the puzzle carefully. Reason internally if needed, but write only the final answer inside \\boxed{} with no trailing text.",
}
SYSTEM_PROMPT = SYSTEM_PROMPTS[PROMPT_EXPERIMENT]
# 'raw' reproduces the accepted baseline. 'boxed' is a metric-aligned reasoning experiment.
TRAIN_TARGET_FORMAT = "boxed"

# Kaggle submission adapter rank must be at most 32.
LORA_R = 8
LORA_ALPHA = 64
LORA_DROPOUT = 0.1
# For Nemotron use ["in_proj", "out_proj"]. For Gemma use ["q_proj", "v_proj"].
# Expanded attention target set for the S4 run.
LORA_TARGET_MODULES = ["in_proj", "out_proj", "q_proj", "k_proj", "v_proj", "o_proj"]
# Effective batch = 16 * 3 = 48. If OOM, use BATCH_SIZE=8 and GRAD_ACCUM_STEPS=6.
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 3
EPOCHS = 1
LEARNING_RATE = 3e-4

# "bf16_fast" is faster/lower memory on A100/H100.
# "fp32_safe" is slower/heavier, but avoids BF16 custom-code dtype issues.
PRECISION_MODE = "bf16_fast"

# Required for Nemotron BF16 training after the index_add dtype error in doc/errors.md.
# This patches only the MoE temporary tensor dtype, not the whole model.
PATCH_NEMOTRON_MOE_DTYPE = True


## 2. Install and import

In [ ]:
!pip install packaging ninja
!pip install -q mamba-ssm --no-build-isolation
!pip uninstall -y -q causal-conv1d
# For any HF basic activities like loading models
# and tokenizers for running inference
# upgrade is a must for the newest Gemma model
!pip install --upgrade datasets
!pip install --upgrade transformers
!pip install --upgrade torchao

# For doing efficient stuff - PEFT
!pip install --upgrade peft
!pip install --upgrade trl
!pip install bitsandbytes
!pip install accelerate

# for logging and visualizing training progress
!pip install tensorboard


In [ ]:

import json
import re
import time
import types
import zipfile

import pandas as pd
import torch
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainerCallback
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from google.colab import drive, files, userdata

if USE_GOOGLE_DRIVE_BACKUP:
    drive.mount("/content/drive", force_remount=False)
    OUTPUT_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
    print("Drive output root:", OUTPUT_PROJECT_ROOT)
else:
    OUTPUT_PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

HF_TOKEN = userdata.get("hftoken")


## 3. Load data

In [ ]:
if not (RAW_DIR / "train.csv").exists() or not (RAW_DIR / "test.csv").exists():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH) as archive:
        archive.extractall(RAW_DIR)

train = pd.read_csv(RAW_DIR / "train.csv")
test = pd.read_csv(RAW_DIR / "test.csv")

assert list(train.columns) == ["id", "prompt", "answer"]
assert list(test.columns) == ["id", "prompt"]
print("train:", train.shape, "test:", test.shape)
display(test)


## 4. Prepare data

In [ ]:
def normalize_answer(value) -> str:
    text = "" if value is None else str(value).strip()
    text = re.sub(r"^`+|`+$", "", text).strip()
    text = re.sub(r"\s+", " ", text)
    return text.strip().strip(".")


def extract_final_answer(text: str) -> str:
    """Extract a final answer using Kaggle's boxed-answer priority.

    The official metric first looks for answers in ``\\boxed{...}``. A
    literal ``}`` can be part of some answers, so this uses the last closing
    brace before the next boxed answer/end of text instead of stopping at the
    first ``}``.
    """
    text = "" if text is None else str(text)
    boxed_starts = list(re.finditer(r"\\boxed\{", text))
    if boxed_starts:
        matches = []
        for idx, match in enumerate(boxed_starts):
            start = match.end()
            end = boxed_starts[idx + 1].start() if idx + 1 < len(boxed_starts) else len(text)
            segment = text[start:end]
            last_brace = segment.rfind("}")
            matches.append(segment[:last_brace] if last_brace != -1 else segment)
        non_empty = [match.strip() for match in matches if match.strip()]
        return normalize_answer(non_empty[-1] if non_empty else matches[-1])

    patterns = [
        r"The final answer is:\s*([^\n]+)",
        r"Final answer is:\s*([^\n]+)",
        r"Final answer\s*[:：]\s*([^\n]+)",
        r"final answer\s*[:：]\s*([^\n]+)",
    ]
    for pattern in patterns:
        matches = re.findall(pattern, text, re.IGNORECASE)
        if matches:
            return normalize_answer(matches[-1])

    number_matches = re.findall(r"-?\d+(?:\.\d+)?", text)
    if number_matches:
        return normalize_answer(number_matches[-1])

    lines = [line.strip() for line in text.splitlines() if line.strip()]
    for line in reversed(lines):
        line = re.sub(r"^(final answer|answer|assistant)\s*[:\-]\s*", "", line, flags=re.I).strip()
        if line:
            return normalize_answer(line.strip("`").strip())
    return normalize_answer(text)


def format_training_answer(answer: str) -> str:
    """Return the assistant target text for SFT examples.

    Use raw to reproduce the accepted 0.62 Colab baseline, where the
    model is trained on the short answer exactly as it appears in train.csv.
    Use boxed only for an explicit experiment aligned to Kaggle's metric
    prompt/extractor.
    """
    answer_text = "" if answer is None else str(answer).strip()
    if TRAIN_TARGET_FORMAT == "raw":
        return answer_text
    if TRAIN_TARGET_FORMAT == "boxed":
        return f"\\boxed{{{normalize_answer(answer_text)}}}"
    raise ValueError(f"Unsupported TRAIN_TARGET_FORMAT: {TRAIN_TARGET_FORMAT!r}")


def build_sft_text(prompt: str, answer: str) -> str:
    return f"System:\n{SYSTEM_PROMPT}\n\nUser:\n{prompt}\n\nAssistant:\n{format_training_answer(answer)}"


def build_prompt(prompt: str) -> str:
    return f"System:\n{SYSTEM_PROMPT}\n\nUser:\n{prompt}\n\nAssistant:\n"


sft = train[["id", "prompt", "answer"]].copy()
sft["text"] = [build_sft_text(p, a) for p, a in zip(sft["prompt"], sft["answer"])]
if EVAL_ROWS and EVAL_ROWS > 0:
    valid = sft.sample(n=min(EVAL_ROWS, len(sft) - 1), random_state=42)
    train_sft = sft.drop(index=valid.index)
else:
    valid = sft.iloc[0:0].copy()
    train_sft = sft.copy()
if TRAIN_ROWS is not None:
    train_sft = train_sft.head(TRAIN_ROWS)
probe_questions = sft.head(PROBE_ROWS).reset_index(drop=True)
PROBE_SET_PATH.parent.mkdir(parents=True, exist_ok=True)
probe_questions.to_csv(PROBE_SET_PATH, index=False)

train_ds = Dataset.from_pandas(train_sft[["text"]], preserve_index=False)
valid_ds = Dataset.from_pandas(valid[["text"]], preserve_index=False) if len(valid) else None
run_config = {
    "experiment_name": EXPERIMENT_NAME,
    "model_name": MODEL_NAME,
    "prompt_experiment": PROMPT_EXPERIMENT,
    "system_prompt": SYSTEM_PROMPT,
    "train_target_format": TRAIN_TARGET_FORMAT,
    "max_seq_length": MAX_SEQ_LENGTH,
    "max_new_tokens": MAX_NEW_TOKENS,
    "train_rows_config": TRAIN_ROWS,
    "train_rows_actual": len(train_sft),
    "eval_rows": len(valid),
    "probe_rows": len(probe_questions),
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "lora_target_modules": LORA_TARGET_MODULES,
    "batch_size": BATCH_SIZE,
    "grad_accum_steps": GRAD_ACCUM_STEPS,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "precision_mode": PRECISION_MODE,
    "use_google_drive_backup": USE_GOOGLE_DRIVE_BACKUP,
    "output_project_root": str(OUTPUT_PROJECT_ROOT),
    "output_dir": str(OUTPUT_DIR),
    "auto_resume_from_checkpoint": AUTO_RESUME_FROM_CHECKPOINT,
    "resume_checkpoint": None if RESUME_CHECKPOINT is None else str(RESUME_CHECKPOINT),
    "force_fresh_run": FORCE_FRESH_RUN,
}
RUN_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
RUN_CONFIG_PATH.write_text(json.dumps(run_config, indent=2), encoding="utf-8")
print("train rows:", len(train_ds), "valid rows:", 0 if valid_ds is None else len(valid_ds))
print("probe rows:", len(probe_questions), "wrote:", PROBE_SET_PATH)
print("wrote run config:", RUN_CONFIG_PATH)


## 5. Load model

In [ ]:
bf16_supported = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
if PRECISION_MODE not in {"bf16_fast", "fp32_safe"}:
    raise ValueError("PRECISION_MODE must be 'bf16_fast' or 'fp32_safe'")
if PRECISION_MODE == "bf16_fast" and not bf16_supported:
    raise RuntimeError("This runtime does not support BF16. Use an A100/H100 or set PRECISION_MODE='fp32_safe'.")

compute_dtype = torch.bfloat16 if PRECISION_MODE == "bf16_fast" else torch.float32
model_dtype = torch.bfloat16 if bf16_supported else torch.float16
trainer_bf16 = PRECISION_MODE == "bf16_fast"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

print("model:", MODEL_NAME)
print("precision mode:", PRECISION_MODE)
print("model dtype:", model_dtype)
print("compute dtype:", compute_dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    torch_dtype=model_dtype,
    device_map={"": 0},
    trust_remote_code=True,
    token=HF_TOKEN,
)
model.config.use_cache = False
model.generation_config.use_cache = False
model.eval()


def _patched_nemotron_moe(self, hidden_states, topk_indices, topk_weights):
    final_hidden_states = torch.zeros_like(hidden_states, dtype=topk_weights.dtype)
    expert_mask = torch.nn.functional.one_hot(topk_indices, num_classes=len(self.experts))
    expert_mask = expert_mask.permute(2, 0, 1)

    for expert_idx in range(len(self.experts)):
        expert = self.experts[expert_idx]
        mask = expert_mask[expert_idx]
        token_indices, weight_indices = torch.where(mask)

        if token_indices.numel() > 0:
            expert_weights = topk_weights[token_indices, weight_indices]
            expert_input = hidden_states[token_indices]
            expert_output = expert(expert_input)
            weighted_output = expert_output * expert_weights.unsqueeze(-1)
            weighted_output = weighted_output.to(final_hidden_states.dtype)
            final_hidden_states.index_add_(0, token_indices, weighted_output)
        else:
            expert_dtype = expert.down_proj.weight.dtype
            dummy_out = expert(torch.zeros_like(hidden_states[0]).unsqueeze(0).to(expert_dtype))
            final_hidden_states = final_hidden_states + dummy_out.to(final_hidden_states.dtype)

    return final_hidden_states.type(hidden_states.dtype)


def patch_nemotron_moe_dtype(current_model):
    patched = 0
    for module in current_model.modules():
        if module.__class__.__name__ == "NemotronHMOE":
            module.moe = types.MethodType(_patched_nemotron_moe, module)
            patched += 1
    print("patched Nemotron MoE dtype modules:", patched)


if PRECISION_MODE == "bf16_fast" and PATCH_NEMOTRON_MOE_DTYPE:
    patch_nemotron_moe_dtype(model)


## 6. Module summary

In [ ]:
def dtype_name(dtype):
    return str(dtype).replace("torch.", "") if dtype is not None else "-"


module_rows = []
for name, module in model.named_modules():
    class_name = module.__class__.__name__
    if "Linear" in class_name or "Conv" in class_name:
        short_name = name.split(".")[-1]
        weight = getattr(module, "weight", None)
        compute_dtype = getattr(module, "compute_dtype", None)
        param_dtypes = sorted({dtype_name(p.dtype) for p in module.parameters(recurse=False)})
        buffer_dtypes = sorted({dtype_name(b.dtype) for b in module.buffers(recurse=False)})
        module_rows.append({
            "module": short_name,
            "class": class_name,
            "weight_dtype": dtype_name(getattr(weight, "dtype", None)),
            "compute_dtype": dtype_name(compute_dtype),
            "param_dtypes": ", ".join(param_dtypes) or "-",
            "buffer_dtypes": ", ".join(buffer_dtypes) or "-",
        })

module_summary = (
    pd.DataFrame(module_rows)
    .groupby(["module", "class", "weight_dtype", "compute_dtype", "param_dtypes", "buffer_dtypes"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

display(module_summary)
print("LoRA targets:", LORA_TARGET_MODULES)


## 7. Baseline inference

In [ ]:
def generate_answers(current_model, rows):
    records = []
    was_training = current_model.training
    current_model.eval()
    device = next(current_model.parameters()).device
    for row in rows.itertuples(index=False):
        inputs = tokenizer(build_prompt(row.prompt), return_tensors="pt", truncation=True, max_length=MAX_SEQ_LENGTH).to(device)
        start = time.time()
        with torch.no_grad():
            output_ids = current_model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        seconds = time.time() - start
        generated_ids = output_ids[0, inputs["input_ids"].shape[1]:]
        generated_token_count = int(generated_ids.numel())
        hit_max_new_tokens = generated_token_count >= MAX_NEW_TOKENS
        raw = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()
        records.append({
            "id": row.id,
            "answer": extract_final_answer(raw),
            "seconds": seconds,
            "generated_tokens": generated_token_count,
            "hit_max_new_tokens": hit_max_new_tokens,
            "raw_output": raw,
        })
    if was_training:
        current_model.train()
    return pd.DataFrame(records)


probe_history = []
if FORCE_FRESH_RUN:
    for path in [PROBE_EVOLUTION_CSV_PATH, PROBE_EVOLUTION_JSONL_PATH]:
        if path.exists():
            path.unlink()
elif PROBE_EVOLUTION_CSV_PATH.exists():
    existing_probe_history = pd.read_csv(PROBE_EVOLUTION_CSV_PATH)
    if len(existing_probe_history):
        probe_history.append(existing_probe_history)
        print("loaded existing probe history:", PROBE_EVOLUTION_CSV_PATH, "rows:", len(existing_probe_history))


def record_probe_snapshot(current_model, step: int, phase: str) -> pd.DataFrame:
    df = generate_answers(current_model, probe_questions).copy()
    df.insert(0, "phase", phase)
    df.insert(1, "step", int(step))
    df["gold"] = probe_questions["answer"].values
    df["match"] = df["gold"].map(normalize_answer) == df["answer"].map(normalize_answer)
    probe_history.append(df)
    history = pd.concat(probe_history, ignore_index=True)
    history.to_csv(PROBE_EVOLUTION_CSV_PATH, index=False)
    with PROBE_EVOLUTION_JSONL_PATH.open("a", encoding="utf-8") as handle:
        for record in df.to_dict("records"):
            handle.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
    print(f"probe snapshot phase={phase} step={step}; wrote {PROBE_EVOLUTION_CSV_PATH}")
    display(df[["phase", "step", "id", "gold", "answer", "match", "seconds", "generated_tokens", "hit_max_new_tokens", "raw_output"]])
    return df


pd.set_option("display.max_colwidth", 240)
baseline_results = record_probe_snapshot(model, step=0, phase="before")


## 8. Train LoRA

In [ ]:
lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=LORA_TARGET_MODULES,
)
if LORA_R > 32:
    raise ValueError("Kaggle submissions require LoRA rank <= 32")

import gc
import math
gc.collect()
torch.cuda.empty_cache()

if FORCE_FRESH_RUN and OUTPUT_DIR.exists():
    raise RuntimeError(
        f"FORCE_FRESH_RUN=True but {OUTPUT_DIR} already exists. "
        "Delete it manually after confirming you do not need its checkpoints."
    )

steps_per_epoch = max(1, math.ceil(len(train_ds) / (BATCH_SIZE * GRAD_ACCUM_STEPS)))
total_train_steps = max(1, math.ceil(steps_per_epoch * float(EPOCHS)))
logging_steps = max(1, total_train_steps // LOG_EVAL_POINTS)
eval_steps = logging_steps
save_steps = max(logging_steps, total_train_steps // SAVE_POINTS)
eval_strategy = "steps" if valid_ds is not None else "no"
print(
    "train steps:", total_train_steps,
    "logging_steps:", logging_steps,
    "eval_steps:", eval_steps if eval_strategy != "no" else None,
    "save_steps:", save_steps,
)

args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    logging_steps=logging_steps,
    logging_first_step=True,
    save_steps=save_steps,
    eval_strategy=eval_strategy,
    eval_steps=eval_steps if eval_strategy != "no" else None,
    report_to="tensorboard",
    fp16=False,
    bf16=trainer_bf16,
)


class ProbeEvolutionCallback(TrainerCallback):
    def on_evaluate(self, args, state, control, model=None, **kwargs):
        if model is not None:
            record_probe_snapshot(model, step=state.global_step, phase="eval")
        return control


trainer = SFTTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    peft_config=lora_config,
    processing_class=tokenizer,
)
trainer.add_callback(ProbeEvolutionCallback())


def latest_checkpoint(output_dir: Path):
    checkpoints = []
    for path in output_dir.glob("checkpoint-*"):
        try:
            step = int(path.name.rsplit("-", 1)[-1])
        except ValueError:
            continue
        checkpoints.append((step, path))
    return max(checkpoints, key=lambda item: item[0])[1] if checkpoints else None


resume_checkpoint = RESUME_CHECKPOINT
if AUTO_RESUME_FROM_CHECKPOINT and resume_checkpoint is None:
    resume_checkpoint = latest_checkpoint(OUTPUT_DIR)

if resume_checkpoint is not None:
    resume_checkpoint = Path(resume_checkpoint)
    print("resuming from checkpoint:", resume_checkpoint)
    trainer.train(resume_from_checkpoint=str(resume_checkpoint))
else:
    print("starting fresh training run")
    trainer.train()
pd.DataFrame(trainer.state.log_history).to_csv(TRAINER_LOG_CSV_PATH, index=False)
print("wrote trainer log:", TRAINER_LOG_CSV_PATH)
trainer.save_model(str(ADAPTER_DIR))


## 9. Compare after training

In [ ]:
after_results = record_probe_snapshot(trainer.model, step=trainer.state.global_step, phase="after")

comparison = baseline_results[["id", "gold", "answer", "raw_output"]].rename(columns={"answer": "before", "raw_output": "before_raw"})
comparison = comparison.merge(
    after_results[["id", "answer", "raw_output"]].rename(columns={"answer": "after", "raw_output": "after_raw"}),
    on="id",
)
comparison["before_match"] = comparison["gold"].map(normalize_answer) == comparison["before"].map(normalize_answer)
comparison["after_match"] = comparison["gold"].map(normalize_answer) == comparison["after"].map(normalize_answer)

display(comparison[["id", "gold", "before", "before_match", "before_raw", "after", "after_match", "after_raw"]])


## 10. Training decision dashboard


In [ ]:
trainer_log = pd.DataFrame(trainer.state.log_history)
trainer_log.to_csv(TRAINER_LOG_CSV_PATH, index=False)
print("wrote trainer log:", TRAINER_LOG_CSV_PATH)

display_cols = [col for col in ["step", "epoch", "loss", "eval_loss", "learning_rate", "grad_norm"] if col in trainer_log.columns]
display(trainer_log[display_cols].tail(30))

try:
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(figsize=(10, 4))
    if {"step", "loss"}.issubset(trainer_log.columns):
        train_loss = trainer_log.dropna(subset=["loss"])
        if len(train_loss):
            ax.plot(train_loss["step"], train_loss["loss"], marker="o", label="train loss")
    if {"step", "eval_loss"}.issubset(trainer_log.columns):
        eval_loss_df = trainer_log.dropna(subset=["eval_loss"])
        if len(eval_loss_df):
            ax.plot(eval_loss_df["step"], eval_loss_df["eval_loss"], marker="o", label="eval loss")
    ax.set_xlabel("optimizer step")
    ax.set_ylabel("loss")
    ax.set_title(EXPERIMENT_NAME)
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()
except Exception as exc:
    print("loss plot skipped:", exc)

probe_evolution = pd.read_csv(PROBE_EVOLUTION_CSV_PATH) if PROBE_EVOLUTION_CSV_PATH.exists() else pd.DataFrame()
if len(probe_evolution):
    probe_evolution["match_bool"] = probe_evolution["match"].astype(str).str.lower().isin(["true", "1"])
    probe_summary = (
        probe_evolution.groupby(["phase", "step"], as_index=False)
        .agg(
            probe_matches=("match_bool", "sum"),
            probe_rows=("match_bool", "count"),
            avg_seconds=("seconds", "mean"),
        )
        .sort_values("step")
    )
    probe_summary["probe_match_rate"] = probe_summary["probe_matches"] / probe_summary["probe_rows"]
    display(probe_summary)

    compact_probe = probe_evolution[["phase", "step", "id", "gold", "answer", "match", "raw_output"]].copy()
    display(compact_probe.tail(PROBE_ROWS * 3))
else:
    probe_summary = pd.DataFrame()
    print("No probe evolution file found yet:", PROBE_EVOLUTION_CSV_PATH)

latest_train_loss = None
if "loss" in trainer_log.columns and trainer_log["loss"].notna().any():
    latest_train_loss = float(trainer_log.dropna(subset=["loss"])["loss"].iloc[-1])

best_eval_loss = None
latest_eval_loss = None
if "eval_loss" in trainer_log.columns and trainer_log["eval_loss"].notna().any():
    eval_losses = trainer_log.dropna(subset=["eval_loss"])
    best_eval_loss = float(eval_losses["eval_loss"].min())
    latest_eval_loss = float(eval_losses["eval_loss"].iloc[-1])

final_probe_matches = None
final_probe_rows = None
if len(probe_evolution):
    final_step = probe_evolution["step"].max()
    final_probe = probe_evolution[probe_evolution["step"] == final_step].copy()
    final_probe["match_bool"] = final_probe["match"].astype(str).str.lower().isin(["true", "1"])
    final_probe_matches = int(final_probe["match_bool"].sum())
    final_probe_rows = int(len(final_probe))

submit_signals = pd.DataFrame([
    {"signal": "latest_train_loss", "value": latest_train_loss, "how_to_use": "Check for obvious failed training or divergence."},
    {"signal": "latest_eval_loss", "value": latest_eval_loss, "how_to_use": "Prefer lower than prior local experiments, but do not overtrust it."},
    {"signal": "best_eval_loss", "value": best_eval_loss, "how_to_use": "Useful if latest eval worsens late in training."},
    {"signal": "final_probe_matches", "value": None if final_probe_matches is None else f"{final_probe_matches}/{final_probe_rows}", "how_to_use": "Small sample only; inspect raw outputs for formatting failures."},
])
display(submit_signals)


## 11. TensorBoard


In [ ]:
%load_ext tensorboard
TENSORBOARD_LOGDIR = str(OUTPUT_DIR)
%tensorboard --logdir $TENSORBOARD_LOGDIR


## 12. Sanity predictions


In [ ]:
sanity_predictions = generate_answers(trainer.model, test)
sanity_submission = sanity_predictions[["id", "answer"]].copy()

assert list(sanity_submission.columns) == ["id", "answer"]
assert sanity_submission["id"].tolist() == test["id"].tolist()
assert sanity_submission["answer"].astype(str).str.len().gt(0).all()

SANITY_CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
sanity_submission.to_csv(SANITY_CSV_PATH, index=False)
sanity_predictions.to_csv(SANITY_RAW_CSV_PATH, index=False)
print("wrote sanity CSV:", SANITY_CSV_PATH)
print("wrote raw sanity CSV:", SANITY_RAW_CSV_PATH)
display(sanity_predictions[["id", "answer", "seconds", "generated_tokens", "hit_max_new_tokens", "raw_output"]])


## 13. Package adapter submission


In [ ]:
def validate_adapter_dir(adapter_dir: Path) -> list[Path]:
    required = ["adapter_config.json", "adapter_model.safetensors"]
    missing = [name for name in required if not (adapter_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(f"missing required adapter files in {adapter_dir}: {missing}")

    config = json.loads((adapter_dir / "adapter_config.json").read_text(encoding="utf-8"))
    rank = int(config.get("r", 0))
    if not 1 <= rank <= 32:
        raise ValueError(f"Kaggle submissions require LoRA rank <= 32; found r={rank}")
    print("adapter rank:", rank)
    return [adapter_dir / name for name in required]


def write_submission_zip(adapter_dir: Path, output_zip: Path) -> None:
    adapter_files = validate_adapter_dir(adapter_dir)
    output_zip.parent.mkdir(parents=True, exist_ok=True)
    if output_zip.exists():
        output_zip.unlink()

    with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in adapter_files:
            archive.write(path, arcname=path.name)

    with zipfile.ZipFile(output_zip) as archive:
        names = sorted(archive.namelist())

    for name in ["adapter_config.json", "adapter_model.safetensors"]:
        assert name in names, names
    print("wrote:", output_zip)
    print("zip contents:", names)


write_submission_zip(ADAPTER_DIR, SUBMISSION_ZIP_PATH)


def write_diagnostics_zip(output_zip: Path) -> None:
    diagnostic_files = [
        SUBMISSION_ZIP_PATH,
        RUN_CONFIG_PATH,
        PROBE_SET_PATH,
        PROBE_EVOLUTION_CSV_PATH,
        PROBE_EVOLUTION_JSONL_PATH,
        TRAINER_LOG_CSV_PATH,
        SANITY_CSV_PATH,
        SANITY_RAW_CSV_PATH,
        ADAPTER_DIR / "adapter_config.json",
    ]
    diagnostic_files.extend(sorted(OUTPUT_DIR.rglob("events.out.tfevents*")))
    output_zip.parent.mkdir(parents=True, exist_ok=True)
    if output_zip.exists():
        output_zip.unlink()

    with zipfile.ZipFile(output_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for path in diagnostic_files:
            if not path.exists():
                print("missing diagnostic file:", path)
                continue
            try:
                arcname = path.relative_to(PROJECT_ROOT)
            except ValueError:
                arcname = path.name
            archive.write(path, arcname=str(arcname))

    with zipfile.ZipFile(output_zip) as archive:
        names = sorted(archive.namelist())
    print("wrote diagnostics:", output_zip)
    display(pd.DataFrame({"diagnostic_file": names}))


write_diagnostics_zip(DIAGNOSTICS_ZIP_PATH)


## 14. Download


In [ ]:
files.download(str(SUBMISSION_ZIP_PATH))
files.download(str(DIAGNOSTICS_ZIP_PATH))
